## Loads data from bronze to silver
- creates temp view `silver_source` with flattened data from `workspace.market_bronze.intraday_responses`
    - duplicates are first resolved by file_modified_at, then source_path
    - timestamp corrected from America/New_York to UTC
- creates table `workspace.market_silver.intraday_candles` with schema identical to `silver_source`
- merges data from `silver_source` into `workspace.market_silver.intraday_candles`
    - update only if temp view values are different than those in table
    - idempotent


In [0]:
-- create temp view with flattened data from bronze layer
-- convert timestamp to UTC, original ts fetched from API has invalid "Z" format
-- add _source_path to help debugging
CREATE OR REPLACE TEMP VIEW silver_source AS
WITH exploded AS (
    SELECT explode(data) as candle, ticker, interval, source_path as _source_path, file_modified_at FROM workspace.market_bronze.intraday_responses
),
flat AS (
    SELECT candle.data as ohlcv, ticker, interval, _source_path, candle.date as date, row_number() OVER (PARTITION BY ticker, interval, candle.date ORDER BY file_modified_at DESC, _source_path DESC) as rn FROM exploded QUALIFY rn = 1
)
SELECT ticker, to_utc_timestamp(to_timestamp(date, "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"), 'America/New_York') as candle_ts, interval, ohlcv.open, ohlcv.high, ohlcv.low, ohlcv.close, ohlcv.volume, ohlcv.is_extended_hours as ext_hours, _source_path from flat;

In [0]:
-- create table with the same schema as temp view
CREATE TABLE IF NOT EXISTS workspace.market_silver.intraday_candles (
    ticker STRING NOT NULL,
    candle_ts TIMESTAMP NOT NULL COMMENT 'candle start time in UTC, converted from America/New_York',
    interval STRING NOT NULL,
    open DOUBLE, 
    high DOUBLE, 
    low DOUBLE, 
    close DOUBLE, 
    volume BIGINT,
    ext_hours BOOLEAN COMMENT 'is candle from pre market/after hours session',
    _source_path STRING
)
COMMENT 'cleaned OHLCV candles, one per (interval, ticker, candle_ts)'

In [0]:
-- merges temp view data into created table on key ticker, interval, candle_ts
-- updates set if any of: open, high, low, close, volume and ext_hours are not matched
MERGE INTO workspace.market_silver.intraday_candles as t USING silver_source as d 
    ON t.ticker = d.ticker AND t.interval = d.interval AND t.candle_ts = d.candle_ts
    WHEN MATCHED AND !(
        (t.open <=> d.open) AND (t.high <=> d.high) AND (t.low <=> d.low) AND (t.close <=> d.close) 
        AND (t.volume <=> d.volume) AND (t.ext_hours <=> d.ext_hours) 
    ) THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *;

## Checks

In [0]:
-- checks if silver temp data (one ingestion) is fully copied from bronze
SELECT
(SELECT COUNT(*) FROM silver_source) AS silver_rows,
(SELECT sum(size(data)) FROM workspace.market_bronze.intraday_responses) AS bronze_candles,
silver_rows = bronze_candles AS equal;

In [0]:
-- checks if data (one ingestion) is properly merged into silver table
SELECT 
    (SELECT COUNT(*) FROM silver_source) as silver_source,
    (SELECT COUNT(*) FROM  workspace.market_silver.intraday_candles) as intraday_candles,
    silver_source = intraday_candles as equal



In [0]:
-- check for duplicates in temp view
SELECT COUNT(*), ticker, interval, candle_ts FROM silver_source GROUP BY ticker, interval, candle_ts HAVING COUNT(*) > 1;